# 3. RAG: answer questions with citations

```
question ──► ① retrieve the 5 most relevant chunks
         ──► ② prompt = rules + chunks + question
         ──► ③ Groq LLM writes the answer, citing chunk IDs
```

The code lives in **`rag_pipeline.py`** (same folder), so this notebook, the evaluation and the app all answer questions the same way.

**LLM:** `openai/gpt-oss-120b`, an open-source model, free on Groq. `temperature=0` keeps answers consistent.

**The prompt's rules** (see `PROMPT` in `rag_pipeline.py`):
1. Answer **only** from the retrieved chunks, so the model doesn't invent facts.
2. If the answer isn't there, say *"I couldn't find that in the reports."*
3. Cite chunk IDs, e.g. `[AAPL_139]`, so every answer can be checked.
4. Table numbers are in millions of US dollars.

## Three search methods

| Method | How it finds chunks |
|---|---|
| `vector` | Embedding search over all chunks (meaning-based) |
| `vector_filter` | Same, but only within the companies named in the question |
| `hybrid_rerank` | ~40 candidates from embedding search **plus** ~40 from keyword search (BM25), then a **reranker** rereads each candidate next to the question and keeps the best 5 |

**Why a reranker?** Embedding search is fast but weak with number-heavy tables. NVIDIA's revenue table
(`Revenue | 215,938 | 130,497 | Up 65%`) ranked around #30, below dozens of paragraphs that only *talk about* revenue.
A reranker (cross-encoder) reads the question and the chunk **together**, which is slower but much more precise,
so it only runs on the shortlist.

## Setup

In [1]:
from pathlib import Path

# Project root = the folder that contains requirements.txt (works no matter where the notebook runs from)
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("Project folder:", ROOT)

Project folder: /Users/os_bhosale/Desktop/Projects/Financial-Filings-Rag


In [2]:
import sys
sys.path.insert(0, str(ROOT / "3_rag"))

from rag_pipeline import Retriever, load_llm, ask, detect_companies

retriever = Retriever()
llm = load_llm()
print("Chunks:", len(retriever.chunks))

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25946.75it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 16661.69it/s]


BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks: 1633


## Ask questions (best method: `hybrid_rerank`)

In [3]:
def show(question, **kwargs):
    result = ask(retriever, llm, question, **kwargs)
    print("Q:", question)
    print("A:", result["answer"])
    print("Sources:", [c["chunk_id"] for c in result["sources"]])
    print()


show("What were Apple's total net sales in fiscal 2026?")
show("What was Amazon's total net sales in 2025?")
show("Who is the CEO of Google?")

Q: What were Apple's total net sales in fiscal 2026?
A: Apple’s total net sales for fiscal 2026 were **$478,834 million**[AAPL_153].
Sources: ['AAPL_153', 'AAPL_25', 'AAPL_168', 'AAPL_30', 'AAPL_139']



Q: What was Amazon's total net sales in 2025?
A: Amazon’s total net sales for 2025 were **$716,924 million**[AMZN_153]
Sources: ['AMZN_136', 'AMZN_153', 'AMZN_267', 'AMZN_181', 'AMZN_154']



Q: Who is the CEO of Google?
A: I couldn't find that in the reports.
Sources: ['AMZN_17', 'AMZN_16', 'AAPL_202', 'MSFT_48', 'AAPL_98']



Google isn't one of the 5 companies, so the chatbot refuses instead of inventing an answer.

## Why the reranker matters

The same question with plain embedding search and with hybrid search + reranker:

In [4]:
question = "What was NVIDIA's total revenue in fiscal year 2026?"
print("Companies detected:", detect_companies(question))
print()
show(question, method="vector")
show(question, method="hybrid_rerank")

Companies detected: ['NVDA']



Q: What was NVIDIA's total revenue in fiscal year 2026?
A: I couldn't find that in the reports.
Sources: ['NVDA_280', 'NVDA_310', 'NVDA_294', 'NVDA_237', 'NVDA_307']



Q: What was NVIDIA's total revenue in fiscal year 2026?
A: NVIDIA reported total revenue of **$215,938 million** for fiscal year 2026[NVDA_230].
Sources: ['NVDA_280', 'NVDA_310', 'NVDA_309', 'NVDA_188', 'NVDA_230']



Plain embedding search misses NVIDIA's revenue table; the reranker finds it.
How much each method helps across many questions is measured in `4_evaluation/evaluation.ipynb`.